# ✈️ AeroSync — Flight Delay Data Wrangling & Analytics

Integrates **2015 US flights + airlines + airports + historical weather** to explore what drives **arrival delays**.

| Stage | What happens |
|---|---|
| 1. Acquire | Kaggle `usdot/flight-delays` (or upload, or a built-in synthetic fallback) |
| 2. Weather | Daily 2015 weather per airport from the free **Open-Meteo archive API** (no key needed) |
| 3. Clean | Validate airports, drop cancelled/diverted, handle nulls, outliers, duplicates |
| 4. Engineer | Time, holiday, congestion, route, distance and weather features |
| 5. Visualize | Interactive Plotly charts |
| 6. ML prep | Leakage-safe features, temporal train/valid/test split, preprocessors, baseline model |
| 7. Dashboard | Streamlit app exposed publicly through **ngrok** |

**How to run:** `Runtime ▸ Run all`. The last cell prints your public dashboard URL.

**Before you run (one-time):** add your ngrok token in Colab ▸ 🔑 *Secrets* as `NGROK_AUTHTOKEN` (enable notebook access), or you will be prompted to paste it.
For real data, optionally add `KAGGLE_USERNAME` and `KAGGLE_KEY` as secrets. Without them the notebook automatically falls back to a synthetic dataset with the same schema so everything still runs end-to-end.

## 1 · Setup

In [1]:
!pip install -q streamlit pyngrok plotly pyarrow kaggle requests scikit-learn

In [2]:
import os, io, json, time, zlib, warnings, subprocess, getpass
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60, "display.width", 200)

# ---------------- CONFIG ----------------
DATA_MODE       = "auto"   # "auto" | "kaggle" | "upload" | "synthetic"
SAMPLE_FRAC     = 0.20     # fraction of the 5.8M real flights to keep (0.20 ~ 1.1M rows, fits free Colab RAM)
TOP_N_AIRPORTS  = 40       # airports (by traffic) for which weather is fetched; flights are restricted to them
DELAY_THRESHOLD = 15       # minutes: arrival delay >= this => IS_DELAYED = 1 (DOT definition)
SYNTHETIC_ROWS  = 400_000
RANDOM_STATE    = 42
# ----------------------------------------

BASE     = Path("/content/aerosync")
RAW_REAL = BASE / "raw"
RAW_SYN  = BASE / "raw_synthetic"
WX_DIR   = BASE / "weather"
PROC     = BASE / "processed"
for p in (RAW_REAL, RAW_SYN, WX_DIR, PROC):
    p.mkdir(parents=True, exist_ok=True)

print("Config OK ->", BASE)

Config OK -> /content/aerosync


## 2 · Data acquisition
Order of attempts in `auto` mode: existing files in `/content/aerosync/raw` → Kaggle API (using Colab secrets) → synthetic fallback.
To use your own files, drop `flights.csv`, `airlines.csv`, `airports.csv` into `/content/aerosync/raw/` (left sidebar ▸ Files), or set `DATA_MODE = "upload"`.

In [3]:
RAW_FILES = ["flights.csv", "airlines.csv", "airports.csv"]

def have_raw(path):
    return all((path / f).exists() for f in RAW_FILES)

def try_kaggle():
    """Download usdot/flight-delays using Colab secrets / env vars / ~/.kaggle/kaggle.json."""
    try:
        from google.colab import userdata
        for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
            try:
                os.environ[k] = userdata.get(k)
            except Exception:
                pass
    except ImportError:
        pass
    has_creds = (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")) or \
                (Path.home() / ".kaggle" / "kaggle.json").exists()
    if not has_creds:
        print("No Kaggle credentials found.")
        return False
    print("Downloading usdot/flight-delays from Kaggle (≈ 200 MB zipped)…")
    r = subprocess.run(["kaggle", "datasets", "download", "-d", "usdot/flight-delays",
                        "-p", str(RAW_REAL), "--unzip"], capture_output=True, text=True)
    print((r.stdout + r.stderr)[-600:])
    return have_raw(RAW_REAL)

def upload_files():
    from google.colab import files
    print("Upload kaggle.json  OR  flights.csv + airlines.csv + airports.csv")
    for name, content in files.upload().items():
        if name == "kaggle.json":
            d = Path.home() / ".kaggle"; d.mkdir(exist_ok=True)
            (d / "kaggle.json").write_bytes(content); os.chmod(d / "kaggle.json", 0o600)
        elif name.lower() in RAW_FILES:
            (RAW_REAL / name.lower()).write_bytes(content)

# ---------- weather simulator (used for synthetic mode and as an API fallback) ----------
WX_COLS = ["temp_mean", "temp_max", "temp_min", "precip_mm", "snow_cm",
           "wind_max_kmh", "gust_max_kmh", "weather_code"]

def synth_weather(code_, lat, lon):
    rng = np.random.default_rng(zlib.crc32(code_.encode()))
    d = pd.date_range("2015-01-01", "2015-12-31", freq="D")
    n = len(d); doy = d.dayofyear.values
    season = -np.cos(2 * np.pi * (doy - 20) / 365)
    mean_t = 34 - 0.55 * abs(lat) + (6 + 0.35 * abs(lat)) * 0.6 * season + rng.normal(0, 3.5, n)
    tmax = mean_t + 5 + rng.normal(0, 1, n)
    tmin = mean_t - 5 + rng.normal(0, 1, n)
    wet = rng.random(n) < 0.25
    precip = np.where(wet, rng.gamma(1.1, 7.0, n), 0.0)
    snow = np.where((tmin < 0) & (precip > 0), precip * 0.9, 0.0)
    wind = rng.gamma(7, 3.5, n) + 6 * (precip > 10)
    gust = wind * rng.uniform(1.3, 1.8, n)
    wcode = np.select([snow >= 1, precip >= 25, precip >= 1], [73, 95, 61], 0)
    return pd.DataFrame({"date": d, "airport": code_, "temp_mean": mean_t, "temp_max": tmax, "temp_min": tmin,
                         "precip_mm": precip, "snow_cm": snow, "wind_max_kmh": wind, "gust_max_kmh": gust,
                         "weather_code": wcode})

# ---------- synthetic generator ----------
SYN_AIRPORTS = [
    ("ATL","Atlanta","GA",33.6367,-84.4281), ("LAX","Los Angeles","CA",33.9425,-118.4081),
    ("ORD","Chicago","IL",41.9786,-87.9048), ("DFW","Dallas-Fort Worth","TX",32.8969,-97.0381),
    ("DEN","Denver","CO",39.8617,-104.6732), ("JFK","New York","NY",40.6398,-73.7789),
    ("SFO","San Francisco","CA",37.6190,-122.3749), ("SEA","Seattle","WA",47.4490,-122.3093),
    ("LAS","Las Vegas","NV",36.0840,-115.1537), ("MCO","Orlando","FL",28.4294,-81.3090),
    ("EWR","Newark","NJ",40.6925,-74.1687), ("CLT","Charlotte","NC",35.2140,-80.9431),
    ("PHX","Phoenix","AZ",33.4343,-112.0116), ("IAH","Houston","TX",29.9844,-95.3414),
    ("MIA","Miami","FL",25.7932,-80.2906), ("BOS","Boston","MA",42.3643,-71.0052),
    ("MSP","Minneapolis","MN",44.8820,-93.2218), ("DTW","Detroit","MI",42.2124,-83.3534),
    ("FLL","Fort Lauderdale","FL",26.0726,-80.1527), ("PHL","Philadelphia","PA",39.8719,-75.2411),
    ("LGA","New York","NY",40.7772,-73.8726), ("BWI","Baltimore","MD",39.1754,-76.6683),
    ("SLC","Salt Lake City","UT",40.7884,-111.9778), ("SAN","San Diego","CA",32.7336,-117.1897),
    ("IAD","Washington","DC",38.9531,-77.4565), ("DCA","Washington","DC",38.8521,-77.0377),
    ("MDW","Chicago","IL",41.7868,-87.7522), ("TPA","Tampa","FL",27.9755,-82.5332),
    ("PDX","Portland","OR",45.5887,-122.5975), ("HNL","Honolulu","HI",21.3187,-157.9225),
]
SYN_AIRLINES = {"UA":"United Air Lines Inc.","AA":"American Airlines Inc.","US":"US Airways Inc.",
                "F9":"Frontier Airlines Inc.","B6":"JetBlue Airways","OO":"Skywest Airlines Inc.",
                "AS":"Alaska Airlines Inc.","NK":"Spirit Air Lines","WN":"Southwest Airlines Co.",
                "DL":"Delta Air Lines Inc.","EV":"Atlantic Southeast Airlines","HA":"Hawaiian Airlines Inc.",
                "MQ":"American Eagle Airlines Inc.","VX":"Virgin America"}
SYN_AL_EFFECT = {"NK":6,"F9":5,"EV":4,"B6":3,"MQ":3,"UA":2,"AA":1,"OO":1,"US":0,"WN":0,"VX":-2,"DL":-3,"AS":-4,"HA":-5}
SYN_AL_WEIGHT = {"WN":18,"DL":15,"AA":12,"OO":10,"UA":10,"EV":7,"B6":5,"MQ":5,"US":4,"AS":3,"NK":3,"F9":2,"VX":1,"HA":1}
SYN_HUBS = {"ATL":10,"ORD":9,"DFW":8,"LAX":8,"DEN":7,"JFK":5,"SFO":5,"CLT":5,"PHX":5,"LAS":5,"IAH":5}

def make_synthetic(n=SYNTHETIC_ROWS, seed=RANDOM_STATE):
    print(f"Generating synthetic dataset ({n:,} flights)…")
    rng = np.random.default_rng(seed)
    ap = pd.DataFrame(SYN_AIRPORTS, columns=["IATA_CODE", "CITY", "STATE", "LATITUDE", "LONGITUDE"])
    ap["AIRPORT"] = ap.CITY + " Airport (" + ap.IATA_CODE + ")"
    ap["COUNTRY"] = "USA"
    A = len(ap)
    wx = pd.concat([synth_weather(r.IATA_CODE, r.LATITUDE, r.LONGITUDE) for r in ap.itertuples()])
    wx["source"] = "synthetic"
    wx.to_csv(RAW_SYN / "weather_synthetic.csv", index=False)
    arr = lambda c: np.vstack([wx[wx.airport == a][c].values for a in ap.IATA_CODE])   # (A, 365)
    P, S, G, TN = arr("precip_mm"), arr("snow_cm"), arr("gust_max_kmh"), arr("temp_min")

    w = np.array([SYN_HUBS.get(a, 3) for a in ap.IATA_CODE], float); w /= w.sum()
    doy = rng.integers(1, 366, n)
    dates = pd.Timestamp("2015-01-01") + pd.to_timedelta(doy - 1, unit="D")
    oi = rng.choice(A, n, p=w); di = rng.choice(A, n, p=w)
    same = oi == di
    di[same] = (di[same] + rng.integers(1, A, same.sum())) % A

    lat, lon = np.radians(ap.LATITUDE.values), np.radians(ap.LONGITUDE.values)
    dlat, dlon = lat[di] - lat[oi], lon[di] - lon[oi]
    h = np.sin(dlat / 2) ** 2 + np.cos(lat[oi]) * np.cos(lat[di]) * np.sin(dlon / 2) ** 2
    dist = (2 * 3958.8 * np.arcsin(np.sqrt(h))).round().clip(80)

    hours = np.arange(5, 24)
    hp = np.array([3,6,8,8,7,7,6,6,7,7,8,8,8,7,6,5,3,2,1], float); hp /= hp.sum()
    hour = rng.choice(hours, n, p=hp)
    sched_dep = hour * 60 + rng.integers(0, 12, n) * 5
    sched_time = (dist / 8 + 35).round()
    sched_arr = (sched_dep + sched_time) % 1440

    codes = list(SYN_AIRLINES)
    aw = np.array([SYN_AL_WEIGHT[c] for c in codes], float); aw /= aw.sum()
    al = rng.choice(codes, n, p=aw)
    al_eff = np.array([SYN_AL_EFFECT[c] for c in al])

    op, os_, og = P[oi, doy - 1], S[oi, doy - 1], G[oi, doy - 1]
    dp, ds = P[di, doy - 1], S[di, doy - 1]
    month = dates.month.values
    base = (-3 + 0.55 * np.maximum(hour - 7, 0) + al_eff + 0.9 * np.minimum(op, 40) + 3 * np.minimum(os_, 10)
            + 0.4 * np.maximum(og - 50, 0) + 3 * np.isin(month, [6, 7, 8]))
    noise = np.where(rng.random(n) < 0.18, rng.exponential(35, n), rng.normal(0, 5, n))
    dep_delay = np.round(base + noise)
    arr_delay = np.round(dep_delay + 0.4 * np.minimum(dp, 40) + 2 * np.minimum(ds, 10)
                         - 0.003 * dist + rng.normal(-3, 8, n))

    p_cancel = np.clip(0.007 + 0.05 * (os_ > 2) + 0.015 * (op > 25) + 0.01 * (og > 70), 0, 0.9)
    cancelled = rng.random(n) < p_cancel
    diverted = (~cancelled) & (rng.random(n) < 0.002)
    ok = ~(cancelled | diverted)

    severity = (op > 10) | (os_ > 1) | (og > 60)
    shape = np.column_stack([np.full(n, 1.5), np.full(n, 0.1), np.full(n, 2.0), np.full(n, 2.5),
                             np.where(severity, 3.5, 0.4)])
    g = rng.gamma(shape); g /= g.sum(1, keepdims=True)
    causes = np.round(g * np.maximum(arr_delay, 0)[:, None])
    has_cause = ok & (arr_delay >= 15)
    causes[~has_cause] = np.nan

    hhmm = lambda m: (np.mod(m, 1440) // 60 * 100 + np.mod(m, 1440) % 60).astype(float)
    taxi = (8 + rng.gamma(2.0, 5.0, n)).round()
    reason = np.where(severity, "B", rng.choice(["A", "C", "D"], n, p=[0.5, 0.45, 0.05]))

    fl = pd.DataFrame({
        "YEAR": 2015, "MONTH": month, "DAY": dates.day.values, "DAY_OF_WEEK": dates.dayofweek.values + 1,
        "AIRLINE": al, "ORIGIN_AIRPORT": ap.IATA_CODE.values[oi], "DESTINATION_AIRPORT": ap.IATA_CODE.values[di],
        "SCHEDULED_DEPARTURE": hhmm(sched_dep).astype(int),
        "DEPARTURE_DELAY": np.where(cancelled, np.nan, dep_delay),
        "TAXI_OUT": np.where(cancelled, np.nan, taxi),
        "SCHEDULED_TIME": sched_time,
        "ELAPSED_TIME": np.where(ok, sched_time + arr_delay - dep_delay, np.nan),
        "DISTANCE": dist,
        "SCHEDULED_ARRIVAL": hhmm(sched_arr).astype(int),
        "ARRIVAL_DELAY": np.where(ok, arr_delay, np.nan),
        "DIVERTED": diverted.astype(int), "CANCELLED": cancelled.astype(int),
        "CANCELLATION_REASON": np.where(cancelled, reason, None),
        "AIR_SYSTEM_DELAY": causes[:, 0], "SECURITY_DELAY": causes[:, 1], "AIRLINE_DELAY": causes[:, 2],
        "LATE_AIRCRAFT_DELAY": causes[:, 3], "WEATHER_DELAY": causes[:, 4],
    })
    fl.to_csv(RAW_SYN / "flights.csv", index=False)
    ap[["IATA_CODE", "AIRPORT", "CITY", "STATE", "COUNTRY", "LATITUDE", "LONGITUDE"]].to_csv(RAW_SYN / "airports.csv", index=False)
    pd.DataFrame({"IATA_CODE": list(SYN_AIRLINES), "AIRLINE": list(SYN_AIRLINES.values())}).to_csv(RAW_SYN / "airlines.csv", index=False)
    print("Synthetic dataset written to", RAW_SYN)

# ---------- decide which data to use ----------
USING_SYNTHETIC = False
if DATA_MODE == "synthetic":
    USING_SYNTHETIC = True
else:
    if DATA_MODE == "upload" and not have_raw(RAW_REAL):
        upload_files()
    if not have_raw(RAW_REAL) and DATA_MODE in ("auto", "kaggle", "upload"):
        try_kaggle()
    if not have_raw(RAW_REAL):
        if DATA_MODE == "auto":
            print("⚠️  Real data not available -> using SYNTHETIC fallback (same schema).")
            USING_SYNTHETIC = True
        else:
            raise FileNotFoundError("flights.csv / airlines.csv / airports.csv not found in " + str(RAW_REAL))

if USING_SYNTHETIC and not have_raw(RAW_SYN):
    make_synthetic()

RAW = RAW_SYN if USING_SYNTHETIC else RAW_REAL
print(f"\nData source: {'SYNTHETIC' if USING_SYNTHETIC else 'REAL (Kaggle usdot/flight-delays)'}  ->  {RAW}")
for f in RAW_FILES:
    print(f"  {f:14s} {(RAW / f).stat().st_size / 1e6:8.1f} MB")

No Kaggle credentials found.
⚠️  Real data not available -> using SYNTHETIC fallback (same schema).
Generating synthetic dataset (400,000 flights)…
Synthetic dataset written to /content/aerosync/raw_synthetic

Data source: SYNTHETIC  ->  /content/aerosync/raw_synthetic
  flights.csv        31.4 MB
  airlines.csv        0.0 MB
  airports.csv        0.0 MB


## 3 · Load raw tables

In [4]:
USECOLS = ["YEAR","MONTH","DAY","DAY_OF_WEEK","AIRLINE","ORIGIN_AIRPORT","DESTINATION_AIRPORT",
           "SCHEDULED_DEPARTURE","DEPARTURE_DELAY","TAXI_OUT","SCHEDULED_TIME","ELAPSED_TIME","DISTANCE",
           "SCHEDULED_ARRIVAL","ARRIVAL_DELAY","DIVERTED","CANCELLED","CANCELLATION_REASON",
           "AIR_SYSTEM_DELAY","SECURITY_DELAY","AIRLINE_DELAY","LATE_AIRCRAFT_DELAY","WEATHER_DELAY"]
STR_COLS = {"AIRLINE": str, "ORIGIN_AIRPORT": str, "DESTINATION_AIRPORT": str, "CANCELLATION_REASON": str}

frac = 1.0 if USING_SYNTHETIC else SAMPLE_FRAC
parts = []
for ch in pd.read_csv(RAW / "flights.csv", usecols=USECOLS, dtype=STR_COLS, chunksize=500_000):
    parts.append(ch.sample(frac=frac, random_state=RANDOM_STATE) if frac < 1 else ch)
flights_raw = pd.concat(parts, ignore_index=True)
del parts

airlines = pd.read_csv(RAW / "airlines.csv")
airports = pd.read_csv(RAW / "airports.csv")

print(f"flights : {flights_raw.shape}   (sample fraction = {frac})")
print(f"airlines: {airlines.shape}   airports: {airports.shape}")
display(flights_raw.head())

# --- raw missing-value audit ---
miss = (flights_raw.isna().mean().mul(100).round(2).rename("missing_%").to_frame()
        .query("`missing_%` > 0").sort_values("missing_%", ascending=False))
print("\nMissing values in raw flights (note: delay-cause columns are only filled for delays ≥ 15 min):")
display(miss)

flights : (400000, 23)   (sample fraction = 1.0)
airlines: (14, 2)   airports: (30, 7)


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_DELAY,TAXI_OUT,SCHEDULED_TIME,ELAPSED_TIME,DISTANCE,SCHEDULED_ARRIVAL,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,2,2,1,WN,SEA,CLT,1200,-5.0,15.0,319.0,307.0,2275.0,1719,-17.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,10,10,6,DL,SFO,JFK,1840,71.0,19.0,358.0,326.0,2580.0,38,39.0,0,0,NaN,13.0,0.0,17.0,8.0,1.0
2,2015,8,27,4,UA,IAD,DTW,1225,17.0,11.0,83.0,75.0,383.0,1348,9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,6,10,3,DL,PHX,DTW,1755,23.0,16.0,244.0,240.0,1668.0,2159,19.0,0,0,NaN,1.0,1.0,6.0,5.0,6.0
4,2015,6,8,1,HA,PHL,BOS,650,1.0,11.0,70.0,67.0,280.0,800,-2.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN



Missing values in raw flights (note: delay-cause columns are only filled for delays ≥ 15 min):


,missing_%
CANCELLATION_REASON,99.04
AIR_SYSTEM_DELAY,79.12
LATE_AIRCRAFT_DELAY,79.12
WEATHER_DELAY,79.12
AIRLINE_DELAY,79.12
SECURITY_DELAY,79.12
ELAPSED_TIME,1.16
ARRIVAL_DELAY,1.16
TAXI_OUT,0.96
DEPARTURE_DELAY,0.96


## 4 · Historical weather
For the busiest airports, pull **daily 2015 weather** (temperature, precipitation, snowfall, wind, gusts, WMO weather code) from the free Open-Meteo archive API at each airport's coordinates.
Results are cached per airport; if an API call fails, that airport falls back to simulated climatology (flagged in the `source` column).

In [5]:
OM_URL = "https://archive-api.open-meteo.com/v1/archive"
OM_VARS = ("temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,"
           "snowfall_sum,wind_speed_10m_max,wind_gusts_10m_max,weather_code")

def fetch_open_meteo(code_, lat, lon, retries=4):
    params = dict(latitude=lat, longitude=lon, start_date="2015-01-01", end_date="2015-12-31",
                  daily=OM_VARS, timezone="auto")
    for k in range(retries):
        try:
            r = requests.get(OM_URL, params=params, timeout=60)
            if r.status_code == 429:
                time.sleep(5 * (k + 1)); continue
            r.raise_for_status()
            d = r.json()["daily"]
            return pd.DataFrame({
                "date": pd.to_datetime(d["time"]), "airport": code_,
                "temp_mean": d["temperature_2m_mean"], "temp_max": d["temperature_2m_max"],
                "temp_min": d["temperature_2m_min"], "precip_mm": d["precipitation_sum"],
                "snow_cm": d["snowfall_sum"], "wind_max_kmh": d["wind_speed_10m_max"],
                "gust_max_kmh": d["wind_gusts_10m_max"], "weather_code": d["weather_code"],
                "source": "open-meteo"})
        except Exception as e:
            err = e; time.sleep(2 * (k + 1))
    print(f"   ! {code_}: API failed ({err}) -> simulated fallback")
    return None

airports_ok = airports.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()

if USING_SYNTHETIC:
    weather = pd.read_csv(RAW / "weather_synthetic.csv", parse_dates=["date"])
    wx_airports = sorted(weather.airport.unique())
else:
    counts = flights_raw.ORIGIN_AIRPORT.value_counts()
    valid = set(airports_ok.IATA_CODE)
    wx_airports = [a for a in counts.index if a in valid][:TOP_N_AIRPORTS]
    coords = airports_ok.set_index("IATA_CODE")
    frames = []
    for i, a in enumerate(wx_airports, 1):
        cache = WX_DIR / f"{a}.csv"
        if cache.exists():
            w = pd.read_csv(cache, parse_dates=["date"])
        else:
            w = fetch_open_meteo(a, coords.at[a, "LATITUDE"], coords.at[a, "LONGITUDE"])
            if w is None:
                w = synth_weather(a, coords.at[a, "LATITUDE"], coords.at[a, "LONGITUDE"]); w["source"] = "synthetic_fallback"
            else:
                time.sleep(0.4)
            w.to_csv(cache, index=False)
        frames.append(w)
        print(f"\r[{i}/{len(wx_airports)}] {a}  ({w.source.iloc[0]})   ", end="")
    weather = pd.concat(frames, ignore_index=True)
    print()

weather = weather.drop_duplicates(["airport", "date"])
print(f"\nWeather table: {weather.shape}, airports: {len(wx_airports)}")
print(weather.source.value_counts().to_string())
display(weather.describe().T[["mean", "min", "max"]].round(2))


Weather table: (10950, 11), airports: 30
source
synthetic    10950


,mean,min,max
date,2015-07-02 00:00:00,2015-01-01 00:00:00,2015-12-31 00:00:00
temp_mean,13.787123,-13.671084,38.189533
temp_max,18.765352,-9.54159,41.923172
temp_min,8.780533,-18.62482,34.504504
precip_mm,1.889905,0.0,58.255358
snow_cm,0.34571,0.0,52.429822
wind_max_kmh,24.896205,2.982057,73.789377
gust_max_kmh,38.635241,4.702279,116.235801
weather_code,14.294155,0.0,95.0


## 5 · Cleaning
Steps: valid 3-letter IATA codes (October 2015 has numeric airport IDs) → airports that have weather → dates/hours → congestion counts (*computed before* removing cancelled flights, because they were still scheduled) → cancellation summary → drop cancelled/diverted → null/duplicate/outlier handling.

In [6]:
log = []
def step(name, d):
    log.append({"step": name, "rows": len(d)})

df0 = flights_raw.copy()
step("raw sample", df0)

# 1) valid IATA airport codes
ok_code = df0.ORIGIN_AIRPORT.str.len().eq(3) & df0.DESTINATION_AIRPORT.str.len().eq(3)
df0 = df0[ok_code]; step("valid 3-letter airport codes", df0)

# 2) both ends have weather
df0 = df0[df0.ORIGIN_AIRPORT.isin(wx_airports) & df0.DESTINATION_AIRPORT.isin(wx_airports)]
step(f"both airports in weather set ({len(wx_airports)})", df0)

# 3) dates + scheduled hours
df0["FL_DATE"] = pd.to_datetime(dict(year=df0.YEAR, month=df0.MONTH, day=df0.DAY))
df0["SCHED_DEP_HOUR"] = (df0.SCHEDULED_DEPARTURE // 100) % 24
df0["SCHED_ARR_HOUR"] = (df0.SCHEDULED_ARRIVAL // 100) % 24

# 4) schedule-based congestion (includes cancelled flights; scaled back up if we sampled)
scale = 1.0 / frac
load_oh = (df0.groupby(["ORIGIN_AIRPORT", "FL_DATE", "SCHED_DEP_HOUR"]).size().mul(scale)
           .rename("ORIGIN_LOAD_HOUR").reset_index())
load_od = (df0.groupby(["ORIGIN_AIRPORT", "FL_DATE"]).size().mul(scale)
           .rename("ORIGIN_LOAD_DAY").reset_index())
load_dh = (df0.groupby(["DESTINATION_AIRPORT", "FL_DATE", "SCHED_ARR_HOUR"]).size().mul(scale)
           .rename("DEST_LOAD_HOUR").reset_index())

# 5) cancellation / diversion summary (before dropping them)
al_name = airlines.set_index("IATA_CODE")["AIRLINE"].to_dict()
reason_map = {"A": "Airline/Carrier", "B": "Weather", "C": "National Air System", "D": "Security"}
summary = {
    "total_scheduled_flights_in_sample": int(len(df0)),
    "sample_fraction": frac,
    "data_source": "synthetic" if USING_SYNTHETIC else "kaggle usdot/flight-delays + open-meteo",
    "cancel_rate_pct": round(df0.CANCELLED.mean() * 100, 3),
    "divert_rate_pct": round(df0.DIVERTED.mean() * 100, 3),
    "cancel_by_airline": {al_name.get(k, k): round(v * 100, 3) for k, v in df0.groupby("AIRLINE").CANCELLED.mean().items()},
    "cancel_by_month": {int(k): round(v * 100, 3) for k, v in df0.groupby("MONTH").CANCELLED.mean().items()},
    "cancel_by_reason": {reason_map.get(k, k): int(v) for k, v in df0.CANCELLATION_REASON.value_counts().items()},
}

# 6) drop cancelled + diverted, null target, impossible values, duplicates
df = df0[(df0.CANCELLED == 0) & (df0.DIVERTED == 0)]; step("not cancelled / diverted", df)
df = df.dropna(subset=["ARRIVAL_DELAY", "DEPARTURE_DELAY", "SCHEDULED_TIME", "DISTANCE"]); step("non-null delays & schedule", df)
df = df[(df.DISTANCE > 0) & (df.SCHEDULED_TIME > 0)]; step("positive distance / duration", df)
df = df[df.ARRIVAL_DELAY.between(-120, 1500)]; step("arrival delay within [-120, 1500] min", df)
df = df.drop_duplicates(); step("de-duplicated", df)

# delay-cause columns exist only for delays >= 15 min -> NaN really means 0 minutes
CAUSES = ["AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY"]
df[CAUSES] = df[CAUSES].fillna(0)

# winsorised target for plots / regression
df["ARR_DELAY_CAPPED"] = df.ARRIVAL_DELAY.clip(-30, 240)
df["IS_DELAYED"] = (df.ARRIVAL_DELAY >= DELAY_THRESHOLD).astype(int)

print(pd.DataFrame(log).assign(dropped=lambda t: -t.rows.diff().fillna(0).astype(int)).to_string(index=False))
print(f"\nOverall: avg arrival delay = {df.ARRIVAL_DELAY.mean():.1f} min | delayed ≥{DELAY_THRESHOLD} min = {df.IS_DELAYED.mean()*100:.1f}% "
      f"| cancel rate = {summary['cancel_rate_pct']}%")

                                 step   rows  dropped
                           raw sample 400000        0
         valid 3-letter airport codes 400000        0
    both airports in weather set (30) 400000        0
             not cancelled / diverted 395367     4633
           non-null delays & schedule 395367        0
         positive distance / duration 395367        0
arrival delay within [-120, 1500] min 395367        0
                        de-duplicated 395367        0

Overall: avg arrival delay = 5.7 min | delayed ≥15 min = 21.1% | cancel rate = 0.958%


## 6 · Integration & feature engineering
Joins airline names, airport geography and **origin + destination weather** (by airport × date), then derives time, holiday, congestion, route, distance and weather-severity features.

In [7]:
from pandas.tseries.holiday import USFederalHolidayCalendar

# ---- joins: airlines & airports ----
df = df.merge(airlines.rename(columns={"IATA_CODE": "AIRLINE", "AIRLINE": "AIRLINE_NAME"}), on="AIRLINE", how="left")
ap_o = airports[["IATA_CODE", "CITY", "STATE", "LATITUDE", "LONGITUDE"]].rename(columns={
    "IATA_CODE": "ORIGIN_AIRPORT", "CITY": "ORIGIN_CITY", "STATE": "ORIGIN_STATE", "LATITUDE": "ORIGIN_LAT", "LONGITUDE": "ORIGIN_LON"})
ap_d = airports[["IATA_CODE", "CITY", "STATE", "LATITUDE", "LONGITUDE"]].rename(columns={
    "IATA_CODE": "DESTINATION_AIRPORT", "CITY": "DEST_CITY", "STATE": "DEST_STATE", "LATITUDE": "DEST_LAT", "LONGITUDE": "DEST_LON"})
df = df.merge(ap_o, on="ORIGIN_AIRPORT", how="left").merge(ap_d, on="DESTINATION_AIRPORT", how="left")

# ---- joins: weather (origin and destination, by date) ----
def wx_side(prefix, key):
    w = weather[["date", "airport"] + WX_COLS].rename(columns={"date": "FL_DATE", "airport": key})
    return w.rename(columns={c: f"{prefix}_{c.upper()}" for c in WX_COLS})
df = df.merge(wx_side("ORIGIN", "ORIGIN_AIRPORT"), on=["ORIGIN_AIRPORT", "FL_DATE"], how="left")
df = df.merge(wx_side("DEST", "DESTINATION_AIRPORT"), on=["DESTINATION_AIRPORT", "FL_DATE"], how="left")

# ---- congestion ----
df = (df.merge(load_oh, on=["ORIGIN_AIRPORT", "FL_DATE", "SCHED_DEP_HOUR"], how="left")
        .merge(load_od, on=["ORIGIN_AIRPORT", "FL_DATE"], how="left")
        .merge(load_dh, on=["DESTINATION_AIRPORT", "FL_DATE", "SCHED_ARR_HOUR"], how="left"))

# ---- calendar features ----
df["IS_WEEKEND"] = df.DAY_OF_WEEK.isin([6, 7]).astype(int)
df["DAY_OF_YEAR"] = df.FL_DATE.dt.dayofyear
df["SEASON"] = df.MONTH.map({12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
                             6: "Summer", 7: "Summer", 8: "Summer", 9: "Fall", 10: "Fall", 11: "Fall"})
hol = USFederalHolidayCalendar().holidays("2014-12-01", "2016-01-31")
near = set()
for h in hol:
    near.update(h + pd.Timedelta(days=k) for k in range(-2, 3))
df["IS_HOLIDAY"] = df.FL_DATE.isin(hol).astype(int)
df["NEAR_HOLIDAY"] = df.FL_DATE.isin(near).astype(int)

df["TIME_BLOCK"] = pd.cut(df.SCHED_DEP_HOUR, [-1, 5, 11, 17, 23], labels=["Night (0-5)", "Morning (6-11)", "Afternoon (12-17)", "Evening (18-23)"])
df["DISTANCE_BAND"] = pd.cut(df.DISTANCE, [0, 500, 1000, 1500, 2500, 1e5], labels=["<500", "500-1000", "1000-1500", "1500-2500", "2500+"])

# ---- route ----
df["ROUTE"] = df.ORIGIN_AIRPORT + "-" + df.DESTINATION_AIRPORT
df["ROUTE_FLIGHTS"] = (df.groupby("ROUTE")["ROUTE"].transform("size") * scale).round()

# ---- weather severity ----
def wx_condition(precip, snow, gust):
    return np.select([snow >= 1, precip >= 10, gust >= 60, precip >= 1],
                     ["Snow", "Heavy rain", "High wind", "Rain"], default="Clear/Dry")
for side in ("ORIGIN", "DEST"):
    p, s, g, t = (df[f"{side}_{c}"] for c in ("PRECIP_MM", "SNOW_CM", "GUST_MAX_KMH", "TEMP_MIN"))
    df[f"WX_COND_{side}"] = wx_condition(p, s, g)
    df[f"{side}_FREEZING"] = (t <= 0).astype(int)
    df[f"{side}_SEVERE_WX"] = ((p >= 10) | (s >= 1) | (g >= 60)).astype(int)
df["ANY_SEVERE_WX"] = ((df.ORIGIN_SEVERE_WX == 1) | (df.DEST_SEVERE_WX == 1)).astype(int)

# ---- memory: low-cardinality strings -> category ----
for c in ["AIRLINE", "AIRLINE_NAME", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "ORIGIN_CITY", "ORIGIN_STATE",
          "DEST_CITY", "DEST_STATE", "SEASON", "WX_COND_ORIGIN", "WX_COND_DEST", "ROUTE"]:
    df[c] = df[c].astype("category")

print("Final shape:", df.shape)
print("Weather join coverage:", f"{df.ORIGIN_PRECIP_MM.notna().mean()*100:.2f}% origin / {df.DEST_PRECIP_MM.notna().mean()*100:.2f}% destination")
display(df.sample(5, random_state=1).T)

Final shape: (395367, 72)
Weather join coverage: 100.00% origin / 100.00% destination


,389699,16961,305613,355655,2501
YEAR,2015,2015,2015,2015,2015
MONTH,4,8,9,2,10
DAY,10,2,13,1,22
DAY_OF_WEEK,5,7,7,7,4
AIRLINE,UA,EV,B6,DL,AA
...,...,...,...,...,...
ORIGIN_SEVERE_WX,0,1,0,0,0
WX_COND_DEST,Clear/Dry,Rain,Clear/Dry,Clear/Dry,Clear/Dry
DEST_FREEZING,0,0,0,0,0
DEST_SEVERE_WX,0,0,0,0,0


In [8]:
# ---- persist processed data + summary for the dashboard ----
DASH_COLS = ["FL_DATE","MONTH","DAY_OF_WEEK","SCHED_DEP_HOUR","TIME_BLOCK","SEASON","AIRLINE","AIRLINE_NAME",
             "ORIGIN_AIRPORT","DESTINATION_AIRPORT","ORIGIN_CITY","ORIGIN_STATE","ORIGIN_LAT","ORIGIN_LON","DEST_STATE",
             "ROUTE","DISTANCE","DISTANCE_BAND","DEPARTURE_DELAY","ARRIVAL_DELAY","ARR_DELAY_CAPPED","IS_DELAYED",
             *CAUSES, "WX_COND_ORIGIN","WX_COND_DEST","ORIGIN_PRECIP_MM","ORIGIN_SNOW_CM","ORIGIN_GUST_MAX_KMH",
             "ORIGIN_TEMP_MIN","IS_HOLIDAY","NEAR_HOLIDAY","ORIGIN_LOAD_HOUR","ANY_SEVERE_WX"]

df.to_parquet(PROC / "flights_processed.parquet", index=False)
df[DASH_COLS].to_parquet(PROC / "flights_dashboard.parquet", index=False)
weather.to_csv(PROC / "weather_daily.csv", index=False)
(PROC / "summary.json").write_text(json.dumps(summary, indent=2))

for f in sorted(PROC.glob("*")):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:8.1f} MB")

flights_dashboard.parquet         9.4 MB
flights_processed.parquet        21.6 MB
summary.json                      0.0 MB
weather_daily.csv                 1.4 MB


## 7 · Exploratory visualizations

In [9]:
DOW = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}
MON = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}

# 1 · Distribution of arrival delay
fig = px.histogram(df, x="ARR_DELAY_CAPPED", nbins=90, title="Arrival delay distribution (capped −30…240 min)",
                   labels={"ARR_DELAY_CAPPED": "Arrival delay (min)"})
fig.add_vline(x=DELAY_THRESHOLD, line_dash="dash", line_color="red", annotation_text="15-min threshold")
fig.show()

# 2 · Hour of day
g = df.groupby("SCHED_DEP_HOUR", observed=True).agg(pct_delayed=("IS_DELAYED", "mean"), avg_delay=("ARRIVAL_DELAY", "mean")).reset_index()
g["pct_delayed"] *= 100
fig = px.bar(g, x="SCHED_DEP_HOUR", y="pct_delayed", color="avg_delay", color_continuous_scale="OrRd",
             title="% delayed by scheduled departure hour (delays snowball through the day)",
             labels={"pct_delayed": "% delayed", "SCHED_DEP_HOUR": "Hour", "avg_delay": "Avg delay (min)"})
fig.show()

# 3 · Month trend
g = df.groupby("MONTH", observed=True).agg(pct_delayed=("IS_DELAYED", "mean"), avg_delay=("ARRIVAL_DELAY", "mean")).reset_index()
g["Month"] = g.MONTH.map(MON); g["pct_delayed"] *= 100
fig = px.line(g, x="Month", y=["pct_delayed", "avg_delay"], markers=True, title="Seasonality: % delayed and mean delay by month")
fig.show()

# 4 · Day-of-week × hour heatmap
pv = df.pivot_table(index="DAY_OF_WEEK", columns="SCHED_DEP_HOUR", values="IS_DELAYED", aggfunc="mean") * 100
pv.index = [DOW[i] for i in pv.index]
fig = px.imshow(pv, aspect="auto", color_continuous_scale="YlOrRd", title="% delayed — weekday × hour", labels=dict(color="% delayed"))
fig.show()

# 5 · Airlines
g = (df.groupby("AIRLINE_NAME", observed=True).agg(pct_delayed=("IS_DELAYED", "mean"), flights=("IS_DELAYED", "size")).reset_index()
       .sort_values("pct_delayed"))
g["pct_delayed"] *= 100
fig = px.bar(g, x="pct_delayed", y="AIRLINE_NAME", orientation="h", color="pct_delayed", color_continuous_scale="RdYlGn_r",
             hover_data=["flights"], title="% of flights delayed ≥15 min by airline")
fig.show()

# 6 · Airport map
g = (df.groupby(["ORIGIN_AIRPORT", "ORIGIN_CITY", "ORIGIN_LAT", "ORIGIN_LON"], observed=True)
       .agg(avg_delay=("ARRIVAL_DELAY", "mean"), flights=("IS_DELAYED", "size")).reset_index().query("flights > 0"))
fig = px.scatter_geo(g, lat="ORIGIN_LAT", lon="ORIGIN_LON", size="flights", color="avg_delay", hover_name="ORIGIN_AIRPORT",
                     color_continuous_scale="RdYlGn_r", scope="usa", title="Origin airports: traffic (size) and mean arrival delay (colour)")
fig.show()

# 7 · Weather impact
order = ["Clear/Dry", "Rain", "Heavy rain", "High wind", "Snow"]
rows = []
for side, lab in (("ORIGIN", "Origin weather"), ("DEST", "Destination weather")):
    t = df.groupby(f"WX_COND_{side}", observed=True).agg(avg_delay=("ARRIVAL_DELAY", "mean"), pct_delayed=("IS_DELAYED", "mean")).reset_index()
    t.columns = ["condition", "avg_delay", "pct_delayed"]; t["side"] = lab; rows.append(t)
t = pd.concat(rows); t["pct_delayed"] *= 100
fig = px.bar(t, x="condition", y="pct_delayed", color="side", barmode="group", category_orders={"condition": order},
             title="Weather condition vs % delayed (origin & destination)")
fig.show()

pb = pd.cut(df.ORIGIN_PRECIP_MM, [-0.01, 0.1, 1, 5, 10, 25, 1e4], labels=["<0.1", "0.1-1", "1-5", "5-10", "10-25", "25+"])
g = df.groupby(pb, observed=True).ARRIVAL_DELAY.mean().reset_index()
fig = px.bar(g, x="ORIGIN_PRECIP_MM", y="ARRIVAL_DELAY", title="Mean arrival delay by origin daily precipitation (mm)",
             labels={"ORIGIN_PRECIP_MM": "Precip (mm)", "ARRIVAL_DELAY": "Mean delay (min)"})
fig.show()

# 8 · Delay causes
c = df[CAUSES].sum().rename(lambda s: s.replace("_DELAY", "").replace("_", " ").title())
fig = px.pie(values=c.values, names=c.index, hole=0.45, title="Share of delay minutes by cause (flights delayed ≥15 min)")
fig.show()

# 9 · Correlations
num = ["ARRIVAL_DELAY", "DEPARTURE_DELAY", "DISTANCE", "SCHED_DEP_HOUR", "ORIGIN_LOAD_HOUR", "ORIGIN_PRECIP_MM",
       "ORIGIN_SNOW_CM", "ORIGIN_GUST_MAX_KMH", "ORIGIN_TEMP_MIN", "DEST_PRECIP_MM", "DEST_SNOW_CM", "IS_HOLIDAY"]
fig = px.imshow(df[num].corr().round(2), text_auto=True, color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto",
                title="Correlation matrix")
fig.show()

## 8 · Machine-learning data preparation
* **Leakage control** — only information known *before departure* is used (schedule, route, weather *forecast proxy*, congestion). `DEPARTURE_DELAY`, taxi times, elapsed/air time and delay causes are excluded.
* **Temporal split** — train Jan–Sep, validation Oct, test Nov–Dec (random splits would leak seasonal/airport-day information).
* **Two preprocessors** are fitted on train only: a tree-friendly ordinal encoder (used for the baseline) and a linear-model pipeline (impute → scale → one-hot) saved with joblib.
* **Targets** — `IS_DELAYED` (classification) and `ARR_DELAY_CAPPED` (regression).

In [10]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve

NUM = ["MONTH", "DAY_OF_WEEK", "DAY_OF_YEAR", "SCHED_DEP_HOUR", "SCHED_ARR_HOUR", "SCHEDULED_TIME", "DISTANCE",
       "IS_WEEKEND", "IS_HOLIDAY", "NEAR_HOLIDAY", "ORIGIN_LOAD_HOUR", "ORIGIN_LOAD_DAY", "DEST_LOAD_HOUR", "ROUTE_FLIGHTS",
       "ORIGIN_TEMP_MEAN", "ORIGIN_TEMP_MIN", "ORIGIN_TEMP_MAX", "ORIGIN_PRECIP_MM", "ORIGIN_SNOW_CM", "ORIGIN_WIND_MAX_KMH", "ORIGIN_GUST_MAX_KMH",
       "DEST_TEMP_MEAN", "DEST_TEMP_MIN", "DEST_TEMP_MAX", "DEST_PRECIP_MM", "DEST_SNOW_CM", "DEST_WIND_MAX_KMH", "DEST_GUST_MAX_KMH",
       "ORIGIN_FREEZING", "DEST_FREEZING", "ORIGIN_SEVERE_WX", "DEST_SEVERE_WX", "ANY_SEVERE_WX"]
CAT = ["AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "TIME_BLOCK", "SEASON", "DISTANCE_BAND", "ORIGIN_STATE", "DEST_STATE"]
TARGETS = ["IS_DELAYED", "ARR_DELAY_CAPPED"]

ml = df[["FL_DATE"] + NUM + CAT + TARGETS].copy()
for c in CAT:
    ml[c] = ml[c].astype(str)

train = ml[ml.FL_DATE.dt.month <= 9]
valid = ml[ml.FL_DATE.dt.month == 10]
test  = ml[ml.FL_DATE.dt.month >= 11]
for name, part in (("train", train), ("valid", valid), ("test", test)):
    part.to_parquet(PROC / f"ml_{name}.parquet", index=False)
    print(f"{name:5s} rows={len(part):>9,}  delayed rate={part.IS_DELAYED.mean()*100:5.1f}%")

# ---- preprocessor for linear models / neural nets ----
linear_pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), CAT),
])
linear_pre.fit(train[NUM + CAT])
joblib.dump({"preprocessor": linear_pre, "num": NUM, "cat": CAT}, PROC / "linear_preprocessor.joblib")
print("Linear design matrix:", linear_pre.transform(valid[NUM + CAT].head(1000)).shape, "(rows sampled)")

# ---- tree baseline ----
tree_pre = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1), CAT),
    ("num", "passthrough", NUM)], verbose_feature_names_out=False)
clf = Pipeline([("pre", tree_pre),
                ("model", HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08, max_leaf_nodes=48, l2_regularization=1.0,
                                                         categorical_features=[True] * len(CAT) + [False] * len(NUM),
                                                         early_stopping=True, random_state=RANDOM_STATE))])
tr = train.sample(min(len(train), 400_000), random_state=RANDOM_STATE)
clf.fit(tr[NUM + CAT], tr.IS_DELAYED)

pv_ = clf.predict_proba(valid[NUM + CAT])[:, 1]
pt_ = clf.predict_proba(test[NUM + CAT])[:, 1]
prec, rec, thr = precision_recall_curve(valid.IS_DELAYED, pv_)
f1s = 2 * prec * rec / (prec + rec + 1e-9)
best_thr = float(thr[np.argmax(f1s[:-1])])

metrics = {
    "model": "HistGradientBoostingClassifier (pre-departure features only)",
    "train_rows": int(len(train)), "valid_rows": int(len(valid)), "test_rows": int(len(test)),
    "n_features": len(NUM) + len(CAT),
    "positive_rate_test": round(float(test.IS_DELAYED.mean()), 4),
    "roc_auc_valid": round(roc_auc_score(valid.IS_DELAYED, pv_), 4),
    "roc_auc_test": round(roc_auc_score(test.IS_DELAYED, pt_), 4),
    "pr_auc_test": round(average_precision_score(test.IS_DELAYED, pt_), 4),
    "f1_test_at_valid_threshold": round(f1_score(test.IS_DELAYED, pt_ >= best_thr), 4),
    "threshold": round(best_thr, 4),
}
print(json.dumps(metrics, indent=2))

vs = valid.sample(min(len(valid), 20_000), random_state=RANDOM_STATE)
pi = permutation_importance(clf, vs[NUM + CAT], vs.IS_DELAYED, scoring="roc_auc", n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.DataFrame({"feature": NUM + CAT, "importance": pi.importances_mean}).sort_values("importance", ascending=False)
imp.to_csv(PROC / "feature_importance.csv", index=False)
(PROC / "ml_metrics.json").write_text(json.dumps(metrics, indent=2))
px.bar(imp.head(15).iloc[::-1], x="importance", y="feature", orientation="h",
       title="Permutation importance (drop in validation ROC-AUC)").show()

train rows=  296,040  delayed rate= 21.1%
valid rows=   33,505  delayed rate= 17.6%
test  rows=   65,822  delayed rate= 22.9%
Linear design matrix: (1000, 162) (rows sampled)
{
  "model": "HistGradientBoostingClassifier (pre-departure features only)",
  "train_rows": 296040,
  "valid_rows": 33505,
  "test_rows": 65822,
  "n_features": 41,
  "positive_rate_test": 0.2289,
  "roc_auc_valid": 0.6827,
  "roc_auc_test": 0.7625,
  "pr_auc_test": 0.6343,
  "f1_test_at_valid_threshold": 0.5263,
  "threshold": 0.1877
}


## 9 · Dashboard (Streamlit)
The next cell writes `app.py`. It reads the processed files, so re-run the pipeline cells above if you change anything.

In [11]:
%%writefile /content/aerosync/app.py
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import streamlit as st

PROC = Path("/content/aerosync/processed")
DOW = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}
MON = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun", 7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}
CAUSES = ["AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY"]

st.set_page_config(page_title="AeroSync", page_icon="✈️", layout="wide")


@st.cache_data(show_spinner="Loading AeroSync data…")
def load_data():
    return pd.read_parquet(PROC / "flights_dashboard.parquet")


def load_json(name):
    p = PROC / name
    return json.loads(p.read_text()) if p.exists() else {}


df = load_data()
summary = load_json("summary.json")
metrics = load_json("ml_metrics.json")

st.title("✈️ AeroSync — 2015 Flight Delay Analytics")
st.caption(f"Data source: {summary.get('data_source', 'n/a')} · sample fraction: {summary.get('sample_fraction', 'n/a')}")

# ---------------- sidebar filters ----------------
st.sidebar.header("Filters")
months = st.sidebar.slider("Month range", 1, 12, (1, 12))
airlines = st.sidebar.multiselect("Airlines", sorted(df.AIRLINE_NAME.dropna().unique()))
states = st.sidebar.multiselect("Origin state", sorted(df.ORIGIN_STATE.dropna().unique()))
conds = st.sidebar.multiselect("Origin weather", sorted(df.WX_COND_ORIGIN.dropna().unique()))

mask = df.MONTH.between(months[0], months[1])
if airlines:
    mask &= df.AIRLINE_NAME.isin(airlines)
if states:
    mask &= df.ORIGIN_STATE.isin(states)
if conds:
    mask &= df.WX_COND_ORIGIN.isin(conds)
f = df[mask]

if f.empty:
    st.warning("No flights match the current filters.")
    st.stop()

# ---------------- KPIs ----------------
k = st.columns(5)
k[0].metric("Flights (sample)", f"{len(f):,}")
k[1].metric("Avg arrival delay", f"{f.ARRIVAL_DELAY.mean():.1f} min")
k[2].metric("Median arrival delay", f"{f.ARRIVAL_DELAY.median():.0f} min")
k[3].metric("% delayed ≥15 min", f"{f.IS_DELAYED.mean() * 100:.1f}%")
k[4].metric("Cancellation rate (all)", f"{summary.get('cancel_rate_pct', float('nan'))}%")

tabs = st.tabs(["Overview", "Time patterns", "Airlines", "Airports", "Weather", "Delay causes", "ML readiness", "Data explorer"])

# ---------------- Overview ----------------
with tabs[0]:
    c1, c2 = st.columns(2)
    m = f.groupby("MONTH", observed=True).agg(avg_delay=("ARRIVAL_DELAY", "mean"), pct_delayed=("IS_DELAYED", "mean")).reset_index()
    m["Month"] = m.MONTH.map(MON)
    m["pct_delayed"] *= 100
    c1.plotly_chart(px.line(m, x="Month", y=["avg_delay", "pct_delayed"], markers=True, title="Monthly trend"), use_container_width=True)
    d = f.groupby("DAY_OF_WEEK", observed=True).IS_DELAYED.mean().mul(100).reset_index()
    d["Day"] = d.DAY_OF_WEEK.map(DOW)
    c2.plotly_chart(px.bar(d, x="Day", y="IS_DELAYED", title="% delayed by weekday", labels={"IS_DELAYED": "% delayed"}), use_container_width=True)
    daily = f.groupby("FL_DATE", observed=True).ARRIVAL_DELAY.mean().rolling(7, min_periods=1).mean().reset_index()
    st.plotly_chart(px.line(daily, x="FL_DATE", y="ARRIVAL_DELAY", title="Daily mean arrival delay (7-day rolling)"), use_container_width=True)
    fig = px.histogram(f, x="ARR_DELAY_CAPPED", nbins=80, title="Arrival delay distribution (capped)")
    st.plotly_chart(fig, use_container_width=True)

# ---------------- Time patterns ----------------
with tabs[1]:
    pv = f.pivot_table(index="DAY_OF_WEEK", columns="SCHED_DEP_HOUR", values="IS_DELAYED", aggfunc="mean") * 100
    pv.index = [DOW[i] for i in pv.index]
    st.plotly_chart(px.imshow(pv, aspect="auto", color_continuous_scale="YlOrRd", title="% delayed — weekday × departure hour"), use_container_width=True)
    h = f.groupby("SCHED_DEP_HOUR", observed=True).agg(pct=("IS_DELAYED", "mean"), flights=("IS_DELAYED", "size")).reset_index()
    h["pct"] *= 100
    st.plotly_chart(px.bar(h, x="SCHED_DEP_HOUR", y="pct", color="flights", title="% delayed by hour (colour = volume)"), use_container_width=True)
    hol = f.groupby("NEAR_HOLIDAY", observed=True).IS_DELAYED.mean().mul(100).rename({0: "Normal days", 1: "±2 days of US holiday"}).reset_index()
    st.plotly_chart(px.bar(hol, x="NEAR_HOLIDAY", y="IS_DELAYED", title="Holiday effect", labels={"IS_DELAYED": "% delayed", "NEAR_HOLIDAY": ""}), use_container_width=True)

# ---------------- Airlines ----------------
with tabs[2]:
    a = (f.groupby("AIRLINE_NAME", observed=True).agg(pct=("IS_DELAYED", "mean"), avg=("ARRIVAL_DELAY", "mean"), flights=("IS_DELAYED", "size"))
         .reset_index().sort_values("pct"))
    a["pct"] *= 100
    c1, c2 = st.columns(2)
    c1.plotly_chart(px.bar(a, x="pct", y="AIRLINE_NAME", orientation="h", color="pct", color_continuous_scale="RdYlGn_r",
                           title="% delayed by airline"), use_container_width=True)
    c2.plotly_chart(px.scatter(a, x="flights", y="avg", text="AIRLINE_NAME", title="Volume vs mean delay",
                               labels={"avg": "Mean delay (min)"}), use_container_width=True)
    if summary.get("cancel_by_airline"):
        cb = pd.Series(summary["cancel_by_airline"]).sort_values().reset_index()
        cb.columns = ["Airline", "Cancel %"]
        st.plotly_chart(px.bar(cb, x="Cancel %", y="Airline", orientation="h", title="Cancellation rate by airline (full data, all months)"), use_container_width=True)

# ---------------- Airports ----------------
with tabs[3]:
    g = (f.groupby(["ORIGIN_AIRPORT", "ORIGIN_CITY", "ORIGIN_LAT", "ORIGIN_LON"], observed=True)
         .agg(avg_delay=("ARRIVAL_DELAY", "mean"), pct=("IS_DELAYED", "mean"), flights=("IS_DELAYED", "size")).reset_index())
    g = g[g.flights > 0]
    g["pct"] *= 100
    st.plotly_chart(px.scatter_geo(g, lat="ORIGIN_LAT", lon="ORIGIN_LON", size="flights", color="avg_delay", hover_name="ORIGIN_AIRPORT",
                                   hover_data=["ORIGIN_CITY", "pct"], color_continuous_scale="RdYlGn_r", scope="usa",
                                   title="Origin airports"), use_container_width=True)
    c1, c2 = st.columns(2)
    c1.subheader("Worst airports (≥200 flights)")
    c1.dataframe(g[g.flights >= 200].sort_values("avg_delay", ascending=False)[["ORIGIN_AIRPORT", "ORIGIN_CITY", "avg_delay", "pct", "flights"]].head(15).round(1), use_container_width=True)
    r = (f.groupby("ROUTE", observed=True).agg(avg_delay=("ARRIVAL_DELAY", "mean"), pct=("IS_DELAYED", "mean"), flights=("IS_DELAYED", "size")).reset_index())
    r["pct"] *= 100
    c2.subheader("Worst routes (≥100 flights)")
    c2.dataframe(r[r.flights >= 100].sort_values("avg_delay", ascending=False).head(15).round(1), use_container_width=True)

# ---------------- Weather ----------------
with tabs[4]:
    order = ["Clear/Dry", "Rain", "Heavy rain", "High wind", "Snow"]
    c1, c2 = st.columns(2)
    w = f.groupby("WX_COND_ORIGIN", observed=True).agg(pct=("IS_DELAYED", "mean"), avg=("ARRIVAL_DELAY", "mean"), flights=("IS_DELAYED", "size")).reset_index()
    w["pct"] *= 100
    c1.plotly_chart(px.bar(w, x="WX_COND_ORIGIN", y="pct", color="avg", category_orders={"WX_COND_ORIGIN": order},
                           title="Origin weather vs % delayed", labels={"WX_COND_ORIGIN": "Condition"}), use_container_width=True)
    w2 = f.groupby("WX_COND_DEST", observed=True).IS_DELAYED.mean().mul(100).reset_index()
    c2.plotly_chart(px.bar(w2, x="WX_COND_DEST", y="IS_DELAYED", category_orders={"WX_COND_DEST": order},
                           title="Destination weather vs % delayed", labels={"WX_COND_DEST": "Condition", "IS_DELAYED": "% delayed"}), use_container_width=True)
    c3, c4 = st.columns(2)
    pb = pd.cut(f.ORIGIN_PRECIP_MM, [-0.01, 0.1, 1, 5, 10, 25, 1e4], labels=["<0.1", "0.1-1", "1-5", "5-10", "10-25", "25+"])
    c3.plotly_chart(px.bar(f.groupby(pb, observed=True).ARRIVAL_DELAY.mean().reset_index(), x="ORIGIN_PRECIP_MM", y="ARRIVAL_DELAY",
                           title="Mean delay by origin precipitation (mm)"), use_container_width=True)
    gb = pd.cut(f.ORIGIN_GUST_MAX_KMH, [0, 30, 40, 50, 60, 70, 500], labels=["<30", "30-40", "40-50", "50-60", "60-70", "70+"])
    c4.plotly_chart(px.bar(f.groupby(gb, observed=True).ARRIVAL_DELAY.mean().reset_index(), x="ORIGIN_GUST_MAX_KMH", y="ARRIVAL_DELAY",
                           title="Mean delay by origin max gust (km/h)"), use_container_width=True)
    tb = pd.cut(f.ORIGIN_TEMP_MIN, [-100, -10, -5, 0, 5, 10, 20, 100], labels=["<-10", "-10..-5", "-5..0", "0..5", "5..10", "10..20", "20+"])
    st.plotly_chart(px.bar(f.groupby(tb, observed=True).ARRIVAL_DELAY.mean().reset_index(), x="ORIGIN_TEMP_MIN", y="ARRIVAL_DELAY",
                           title="Mean delay by origin daily min temperature (°C)"), use_container_width=True)

# ---------------- Delay causes ----------------
with tabs[5]:
    c = f[CAUSES].sum().rename(lambda s: s.replace("_DELAY", "").replace("_", " ").title())
    c1, c2 = st.columns(2)
    c1.plotly_chart(px.pie(values=c.values, names=c.index, hole=0.45, title="Share of delay minutes by cause"), use_container_width=True)
    mc = f.groupby("MONTH", observed=True)[CAUSES].sum().reset_index()
    mc["Month"] = mc.MONTH.map(MON)
    c2.plotly_chart(px.bar(mc.melt(id_vars=["MONTH", "Month"], value_vars=CAUSES), x="Month", y="value", color="variable",
                           title="Delay minutes by cause and month", labels={"value": "Minutes"}), use_container_width=True)
    if summary.get("cancel_by_reason"):
        cr = pd.Series(summary["cancel_by_reason"]).reset_index()
        cr.columns = ["Reason", "Cancelled flights"]
        st.plotly_chart(px.bar(cr, x="Reason", y="Cancelled flights", title="Cancellation reasons (sample)"), use_container_width=True)

# ---------------- ML readiness ----------------
with tabs[6]:
    st.markdown("**Leakage-safe features** (schedule, route, congestion, weather) with a temporal split: train Jan–Sep · valid Oct · test Nov–Dec.")
    if metrics:
        mm = st.columns(4)
        mm[0].metric("ROC-AUC (test)", metrics["roc_auc_test"])
        mm[1].metric("PR-AUC (test)", metrics["pr_auc_test"])
        mm[2].metric("F1 (test)", metrics["f1_test_at_valid_threshold"])
        mm[3].metric("Delayed rate (test)", f"{metrics['positive_rate_test'] * 100:.1f}%")
        st.json(metrics)
    imp_path = PROC / "feature_importance.csv"
    if imp_path.exists():
        imp = pd.read_csv(imp_path).head(15).iloc[::-1]
        st.plotly_chart(px.bar(imp, x="importance", y="feature", orientation="h", title="Permutation importance (ROC-AUC drop)"), use_container_width=True)

# ---------------- Data explorer ----------------
with tabs[7]:
    st.write(f"{len(f):,} rows match the filters.")
    st.dataframe(f.sample(min(len(f), 2000), random_state=1), use_container_width=True)
    st.download_button("Download filtered data (first 50k rows, CSV)", f.head(50_000).to_csv(index=False).encode(), "aerosync_filtered.csv", "text/csv")

Writing /content/aerosync/app.py


## 10 · Launch the dashboard through ngrok
Put your token in Colab ▸ 🔑 Secrets as `NGROK_AUTHTOKEN` (recommended) or paste it when prompted.
ngrok's free tier shows a one-time "Visit Site" interstitial — click it to reach the dashboard.

In [15]:
from pyngrok import ngrok, conf
import os, subprocess, time, getpass, requests

token = None
try:
    from google.colab import userdata
    token = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    pass

token = token or os.environ.get("NGROK_AUTHTOKEN") or getpass.getpass("Paste your ngrok authtoken: ").strip()

# Clean the token if the user pasted the entire "ngrok config add-authtoken <token>" command
if token.startswith("ngrok config add-authtoken"):
    token = token.split("add-authtoken")[-1].strip()

try:
    ngrok.set_auth_token(token)
except Exception as e:
    if "already set" in str(e):
        print("Authtoken is already configured in the ngrok.yml config file. Continuing...")
    else:
        raise e

# clean restart
ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(1)

log_file = open("/content/streamlit.log", "w")
proc = subprocess.Popen(
    ["streamlit", "run", "/content/aerosync/app.py", "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false", "--browser.gatherUsageStats", "false"],
    stdout=log_file, stderr=subprocess.STDOUT)

for _ in range(30):                       # wait until Streamlit answers
    try:
        if requests.get("http://localhost:8501/_stcore/health", timeout=2).text.strip() == "ok":
            break
    except Exception:
        pass
    time.sleep(1)

tunnel = ngrok.connect(8501, "http")
print("\n  AeroSync dashboard is live:\n   ", tunnel.public_url)
print("\n(Click 'Visit Site' on the ngrok warning page. Keep this notebook running to keep the dashboard online.)")

Paste your ngrok authtoken: ··········

  AeroSync dashboard is live:
    https://washhouse-jacket-nutcase.ngrok-free.dev

(Click 'Visit Site' on the ngrok warning page. Keep this notebook running to keep the dashboard online.)


In [ ]:
# Optional: troubleshooting / shutdown helpers
# print(open("/content/streamlit.log").read()[-3000:])   # Streamlit logs
# ngrok.kill(); proc.terminate()                         # stop dashboard + tunnel